# Figure 5 RSSI CDF Pipeline

This notebook scans the office AP-sweep artifacts, extracts the final method snapshots, renders one coverage map per method, exports the RSSI samples used for the CDF, and writes a summary JSON with the extracted configuration and statistics.

In [9]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any, Mapping, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sionna.rt import RadioMapSolver

from reflector_position.metrics import POWER_EPSILON, rss_to_dbm
from reflector_position.optimizers.memetic.memetic_plotting import (
    _apply_snapshot_to_scene,
    _render_coverage_snapshot,
)
from reflector_position.scene_setup import setup_building_floor_scene


repo_root = Path("/home/hieule/research/reflector-position")
artifacts_dir = Path(
    "/home/hieule/research/reflector-position/tmp_comparison_results2/office/per_trial_runs/aps_03_seed_0042/artifacts"
)
run_dir = artifacts_dir.parent

output_root = repo_root / "output" / "figure5"
coverage_dir = output_root / "coverage_maps"
output_root.mkdir(parents=True, exist_ok=True)
coverage_dir.mkdir(parents=True, exist_ok=True)


def _load_json(path: Path) -> Optional[dict[str, Any]]:
    if not path.exists():
        return None
    return json.loads(path.read_text())


def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def _is_sequence(value: Any) -> bool:
    return isinstance(value, Sequence) and not isinstance(value, (str, bytes, bytearray))


def _as_float(value: Any) -> Optional[float]:
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(numeric):
        return None
    return numeric


def _first_present(container: Mapping[str, Any], *keys: str) -> Any:
    for key in keys:
        if key in container:
            value = container.get(key)
            if value is not None:
                return value
    return None


def _coerce_triplet(raw_value: Any, fallback_z: Optional[float] = None) -> Optional[list[float]]:
    if not _is_sequence(raw_value) or len(raw_value) < 2:
        return None
    try:
        x_value = float(raw_value[0])
        y_value = float(raw_value[1])
        z_value = float(raw_value[2]) if len(raw_value) >= 3 else float(3.8 if fallback_z is None else fallback_z)
    except (TypeError, ValueError):
        return None
    return [x_value, y_value, z_value]


def _normalize_positions(raw_positions: Any, fallback_z: Optional[float]) -> Optional[list[list[float]]]:
    if not _is_sequence(raw_positions) or len(raw_positions) == 0:
        return None
    normalized: list[list[float]] = []
    for raw_position in raw_positions:
        triplet = _coerce_triplet(raw_position, fallback_z=fallback_z)
        if triplet is not None:
            normalized.append(triplet)
    return normalized or None


def _normalize_directions(raw_directions: Any) -> Optional[list[list[float]]]:
    if not _is_sequence(raw_directions) or len(raw_directions) == 0:
        return None
    normalized: list[list[float]] = []
    for raw_direction in raw_directions:
        if not _is_sequence(raw_direction) or len(raw_direction) < 2:
            continue
        try:
            dx_value = float(raw_direction[0])
            dy_value = float(raw_direction[1])
            dz_value = float(raw_direction[2]) if len(raw_direction) >= 3 else 0.0
        except (TypeError, ValueError):
            continue
        normalized.append([dx_value, dy_value, dz_value])
    return normalized or None


def _normalize_reflector(container: Mapping[str, Any]) -> Optional[dict[str, Any]]:
    raw_reflector = container.get("reflector")
    reflector: dict[str, Any] = {}

    if isinstance(raw_reflector, Mapping):
        for key in ("u", "v"):
            if key in raw_reflector:
                value = _as_float(raw_reflector.get(key))
                if value is not None:
                    reflector[key] = value
        if "target" in raw_reflector:
            target = _coerce_triplet(raw_reflector.get("target"), fallback_z=1.5)
            if target is not None:
                reflector["target"] = target
        elif all(key in raw_reflector for key in ("focal_x", "focal_y", "focal_z")):
            target = _coerce_triplet(
                [raw_reflector.get("focal_x"), raw_reflector.get("focal_y"), raw_reflector.get("focal_z")],
                fallback_z=1.5,
            )
            if target is not None:
                reflector["target"] = target
        return reflector or None

    u_value = _as_float(container.get("reflector_u"))
    v_value = _as_float(container.get("reflector_v"))
    if u_value is not None:
        reflector["u"] = u_value
    if v_value is not None:
        reflector["v"] = v_value

    target = _first_present(container, "reflector_target", "initial_focal_point", "focal_point")
    if target is None and all(key in container for key in ("focal_x", "focal_y", "focal_z")):
        target = [container["focal_x"], container["focal_y"], container["focal_z"]]
    target_triplet = _coerce_triplet(target, fallback_z=1.5)
    if target_triplet is not None:
        reflector["target"] = target_triplet

    return reflector or None


def _extract_direct_snapshot(container: Mapping[str, Any], fixed_z: Optional[float]) -> Optional[dict[str, Any]]:
    raw_positions = _first_present(container, "positions", "ap_positions", "best_positions", "final_positions")
    raw_directions = _first_present(container, "directions", "ap_directions", "best_directions", "final_directions")
    raw_reflector = container.get("reflector")

    if raw_positions is None and raw_directions is None and raw_reflector is None:
        return None

    positions = _normalize_positions(raw_positions, fallback_z=fixed_z)
    if positions is None:
        return None

    snapshot: dict[str, Any] = {"positions": positions}
    directions = _normalize_directions(raw_directions)
    if directions is not None:
        snapshot["directions"] = directions
    reflector = _normalize_reflector(container)
    if reflector is not None:
        snapshot["reflector"] = reflector
    return snapshot


def _extract_snapshot(container: Any, fixed_z: Optional[float], depth: int = 0) -> Optional[dict[str, Any]]:
    if not isinstance(container, Mapping) or depth > 6:
        return None

    direct_snapshot = _extract_direct_snapshot(container, fixed_z=fixed_z)
    if direct_snapshot is not None:
        return direct_snapshot

    for key in ("global_best_result", "results", "best_configuration", "final_configuration"):
        nested = container.get(key)
        if isinstance(nested, Mapping):
            snapshot = _extract_snapshot(nested, fixed_z=fixed_z, depth=depth + 1)
            if snapshot is not None:
                return snapshot

    for key in ("hall_of_fame", "seeds"):
        nested = container.get(key)
        if _is_sequence(nested):
            for item in nested:
                snapshot = _extract_snapshot(item, fixed_z=fixed_z, depth=depth + 1)
                if snapshot is not None:
                    return snapshot

    return None


def _discover_method_payloads(directory: Path) -> dict[str, dict[str, Any]]:
    payloads: dict[str, dict[str, Any]] = {}
    for path in sorted(directory.glob("*_results.json")):
        method_name = path.name[: -len("_results.json")]
        payloads[method_name] = json.loads(path.read_text())
    return payloads


def _resolve_render_config(
    base_config: Mapping[str, Any],
    experiment_summary: Mapping[str, Any] | None,
    plot_data: Mapping[str, Any] | None,
) -> tuple[dict[str, Any], dict[str, Any], dict[str, Any]]:
    scene_config: dict[str, Any] = {}
    raw_scene_config = base_config.get("scene_config")
    if isinstance(raw_scene_config, Mapping):
        scene_config.update(dict(raw_scene_config))

    raw_visualization_scene_config = base_config.get("visualization_scene_config")
    if isinstance(raw_visualization_scene_config, Mapping):
        scene_config.update(dict(raw_visualization_scene_config))

    for key in (
        "position_bounds",
        "num_aps",
        "fixed_z",
        "reflector_enabled",
        "focal_z",
        "tx_positions",
        "tx_power_dbm",
        "rx_position",
        "wall_top_left",
        "wall_bottom_right",
        "focal_point",
        "device",
    ):
        if key in base_config and base_config.get(key) is not None:
            scene_config[key] = base_config.get(key)

    if isinstance(plot_data, Mapping) and plot_data.get("position_bounds") is not None:
        scene_config["position_bounds"] = plot_data.get("position_bounds")

    camera: dict[str, Any] = {}
    raw_camera = base_config.get("camera")
    if isinstance(raw_camera, Mapping):
        camera.update(dict(raw_camera))

    render_settings: dict[str, Any] = {}
    raw_render_settings = base_config.get("coverage_plot_settings")
    if isinstance(raw_render_settings, Mapping):
        render_settings.update(dict(raw_render_settings))

    if isinstance(experiment_summary, Mapping):
        config_overrides = experiment_summary.get("coverage_plot_settings")
        if isinstance(config_overrides, Mapping):
            render_settings.update(dict(config_overrides))

    if "scene_path" not in scene_config or not str(scene_config.get("scene_path", "")).strip():
        raise ValueError("scene_path could not be resolved from the base configuration")

    return scene_config, camera, render_settings


def _empirical_cdf(values: Sequence[float]) -> tuple[np.ndarray, np.ndarray]:
    sorted_values = np.sort(np.asarray(values, dtype=float))
    probabilities = np.arange(1, sorted_values.size + 1, dtype=float) / float(sorted_values.size)
    return sorted_values, probabilities


def _render_coverage_map(
    scene_config: Mapping[str, Any],
    snapshot: Mapping[str, Any],
    save_path: Path,
    samples_per_tx: int,
    max_depth: int,
    resolution: tuple[int, int],
    camera_position: tuple[float, float, float],
    camera_look_at: tuple[float, float, float],
) -> Optional[str]:
    return _render_coverage_snapshot(
        scene_config=scene_config,
        snapshot=snapshot,
        save_path=save_path,
        samples_per_tx=samples_per_tx,
        max_depth=max_depth,
        resolution=resolution,
        camera_position=camera_position,
        camera_look_at=camera_look_at,
    )


def _collect_rssi_samples(
    scene_config: Mapping[str, Any],
    snapshot: Mapping[str, Any],
    samples_per_tx: int,
    max_depth: int,
) -> tuple[np.ndarray, dict[str, float]]:
    loaded = setup_building_floor_scene(
        scene_path=str(scene_config["scene_path"]),
        frequency=scene_config.get("frequency", 6e9),
        tx_positions=scene_config.get("tx_positions", None),
        num_aps=scene_config.get("num_aps", None),
        position_bounds=scene_config.get("position_bounds", None),
        tx_power_dbm=scene_config.get("tx_power_dbm", 5.0),
        rx_position=scene_config.get("rx_position", (16.0, 16.5, 1.5)),
        reflector_enabled=scene_config.get("reflector_enabled", False),
        reflector_size=tuple(scene_config.get("reflector_size", (2.0, 2.0))),
        wall_top_left=scene_config.get("wall_top_left", None),
        wall_bottom_right=scene_config.get("wall_bottom_right", None),
        focal_point=scene_config.get("focal_point", None),
        device=scene_config.get("device", "cuda"),
    )
    if isinstance(loaded, tuple) and len(loaded) == 2:
        scene, reflector_controller = loaded
    else:
        scene, reflector_controller = loaded, None

    _apply_snapshot_to_scene(scene, reflector_controller, snapshot)

    solver = RadioMapSolver()
    radio_map = solver(
        scene,
        cell_size=(1.0, 1.0),
        samples_per_tx=int(samples_per_tx),
        max_depth=int(max_depth),
        refraction=True,
        diffraction=True,
    )

    raw_rss = np.asarray(radio_map.rss)
    rss_tensor = torch.as_tensor(raw_rss, dtype=torch.float32)
    valid_mask = torch.isfinite(rss_tensor) & (rss_tensor > POWER_EPSILON)
    valid_dbm = rss_to_dbm(rss_tensor[valid_mask]).detach().cpu().numpy()

    if valid_dbm.size == 0:
        metrics = {
            "sample_count": 0.0,
            "mean_rssi_dbm": math.nan,
            "median_rssi_dbm": math.nan,
            "min_rssi_dbm": math.nan,
            "p5_rssi_dbm": math.nan,
            "p95_rssi_dbm": math.nan,
        }
        return valid_dbm, metrics

    metrics = {
        "sample_count": float(valid_dbm.size),
        "mean_rssi_dbm": float(np.mean(valid_dbm)),
        "median_rssi_dbm": float(np.median(valid_dbm)),
        "min_rssi_dbm": float(np.min(valid_dbm)),
        "p5_rssi_dbm": float(np.percentile(valid_dbm, 5)),
        "p95_rssi_dbm": float(np.percentile(valid_dbm, 95)),
    }
    return valid_dbm, metrics


In [10]:
base_config_path = repo_root / "configs" / "run_experiments_cuda_office.json"
base_config = _load_json(base_config_path)
if base_config is None:
    raise FileNotFoundError(base_config_path)

experiment_summary = _load_json(artifacts_dir / "experiment_summary.json") or {}
plot_data = _load_json(artifacts_dir / "plot_data.json") or {}
method_summary_path = artifacts_dir / "method_summary.csv"
method_summary_df = pd.read_csv(method_summary_path)
method_summary_lookup = {
    str(row["method"]): row
    for row in method_summary_df.to_dict(orient="records")
    if row.get("method") is not None
}
method_payloads = _discover_method_payloads(artifacts_dir)
raw_scene_config, camera_config, render_settings = _resolve_render_config(
    base_config=base_config,
    experiment_summary=experiment_summary,
    plot_data=plot_data,
)


def _build_common_scene_config(scene_payload: Mapping[str, Any]) -> dict[str, Any]:
    tx_positions = _normalize_positions(scene_payload.get("tx_positions"), fallback_z=3.8)
    focal_point = _coerce_triplet(scene_payload.get("focal_point"), fallback_z=1.5)
    rx_position = _coerce_triplet(scene_payload.get("rx_position"), fallback_z=1.5)

    position_bounds = scene_payload.get("position_bounds")
    if not isinstance(position_bounds, Mapping):
        position_bounds = None

    reflector_size = scene_payload.get("reflector_size")
    if _is_sequence(reflector_size) and len(reflector_size) >= 2:
        reflector_size_tuple = (float(reflector_size[0]), float(reflector_size[1]))
    else:
        reflector_size_tuple = (2.0, 2.0)

    wall_top_left = _coerce_triplet(scene_payload.get("wall_top_left"), fallback_z=3.0)
    wall_bottom_right = _coerce_triplet(scene_payload.get("wall_bottom_right"), fallback_z=1.0)

    num_aps = scene_payload.get("num_aps")
    if num_aps is None and tx_positions is not None:
        num_aps = len(tx_positions)

    fixed_z = _as_float(scene_payload.get("fixed_z"))
    if fixed_z is None and tx_positions and len(tx_positions[0]) >= 3:
        fixed_z = float(tx_positions[0][2])
    if fixed_z is None:
        fixed_z = 3.8

    return {
        "scene_path": str(scene_payload.get("scene_path", "")).strip(),
        "frequency": float(scene_payload.get("frequency", 5.18e9)),
        "tx_positions": tx_positions,
        "num_aps": int(num_aps) if num_aps is not None else None,
        "position_bounds": dict(position_bounds) if position_bounds is not None else None,
        "tx_power_dbm": float(scene_payload.get("tx_power_dbm", 5.0)),
        "rx_position": tuple(rx_position if rx_position is not None else [16.0, 16.5, 1.5]),
        "reflector_enabled": bool(scene_payload.get("reflector_enabled", False)),
        "reflector_size": reflector_size_tuple,
        "wall_top_left": wall_top_left,
        "wall_bottom_right": wall_bottom_right,
        "focal_point": focal_point,
        "device": str(scene_payload.get("device", "cuda")),
        "fixed_z": float(fixed_z),
    }


def _build_unified_snapshot(raw_snapshot: Mapping[str, Any], method_name: str, seed: Optional[int]) -> dict[str, Any]:
    positions = _normalize_positions(raw_snapshot.get("positions"), fallback_z=3.8)
    if positions is None:
        positions = []

    directions = _normalize_directions(raw_snapshot.get("directions"))
    if directions is None:
        directions = []

    raw_reflector = raw_snapshot.get("reflector") if isinstance(raw_snapshot.get("reflector"), Mapping) else {}
    reflector_target = _coerce_triplet(raw_reflector.get("target"), fallback_z=1.5)
    reflector_payload = {
        "u": _as_float(raw_reflector.get("u")),
        "v": _as_float(raw_reflector.get("v")),
        "target": reflector_target,
    }

    return {
        "positions": positions,
        "directions": directions,
        "reflector": reflector_payload,
        "method_name": method_name,
        "seed": seed,
        "ap_count": len(positions),
    }


common_scene_config = _build_common_scene_config(raw_scene_config)
if not common_scene_config["scene_path"]:
    raise ValueError("scene_path could not be resolved from config file run_experiments_cuda_office.json")

camera_position = tuple(float(value) for value in camera_config.get("position", (20.0, 20.0, 70.0)))
camera_look_at = tuple(float(value) for value in camera_config.get("look_at", (20.0, 20.1, 1.5)))
samples_per_tx = int(render_settings.get("samples_per_tx", 1_000_000))
max_depth = int(render_settings.get("max_depth", 13))
resolution_raw = render_settings.get("resolution", (1200, 900))
if _is_sequence(resolution_raw) and len(resolution_raw) >= 2:
    resolution = (int(resolution_raw[0]), int(resolution_raw[1]))
else:
    resolution = (1200, 900)

run_seed = None
run_dir_name = run_dir.name
if "seed_" in run_dir_name:
    seed_fragment = run_dir_name.split("seed_", 1)[-1]
    try:
        run_seed = int(seed_fragment)
    except ValueError:
        run_seed = None

scenario_name = "office"
ap_count_for_name = int(common_scene_config.get("num_aps") or 0)
aps_tag = f"aps{ap_count_for_name}" if ap_count_for_name > 0 else "aps_unknown"

ranked_methods = experiment_summary.get("analysis", {}).get("ranked_methods", []) if isinstance(experiment_summary, Mapping) else []
method_order = [str(item.get("method")) for item in ranked_methods if isinstance(item, Mapping) and item.get("method") is not None]
if not method_order:
    method_order = list(method_summary_lookup.keys())
for method_name in sorted(method_payloads):
    if method_name not in method_order:
        method_order.append(method_name)

method_to_file_token = {
    "pso_gd": "pso_gd",
    "memetic": "ga_gd",
    "random_gd": "random_gd",
    "weighted_kmeans": "weighted_kmeans",
    "kmeans": "kmeans",
    "random": "random",
}
method_to_display_name = {
    "pso_gd": "PSO+GD",
    "memetic": "GA+GD",
    "random_gd": "random+GD",
    "weighted_kmeans": "weighted k-means",
    "kmeans": "k-means",
    "random": "random",
}
method_colors = {
    "pso_gd": "#1f77b4",
    "memetic": "#ff7f0e",
    "random_gd": "#2ca02c",
    "weighted_kmeans": "#d62728",
    "kmeans": "#9467bd",
    "random": "#8c564b",
}

all_sample_rows: list[dict[str, Any]] = []
method_records: dict[str, dict[str, Any]] = {}
missing_snapshot_methods: list[str] = []
render_errors: dict[str, str] = {}

for method_name in method_order:
    payload = method_payloads.get(method_name)
    if payload is None:
        continue

    fixed_z = _as_float(common_scene_config.get("fixed_z"))
    if fixed_z is None:
        fixed_z = 3.8

    extracted_snapshot = _extract_snapshot(payload, fixed_z=fixed_z)
    if extracted_snapshot is None:
        missing_snapshot_methods.append(method_name)
        render_errors[method_name] = "Missing valid final snapshot"
        continue

    snapshot = _build_unified_snapshot(extracted_snapshot, method_name=method_name, seed=run_seed)

    method_scene_config = dict(common_scene_config)
    method_scene_config["num_aps"] = int(snapshot["ap_count"])
    method_scene_config["reflector_enabled"] = bool(common_scene_config["reflector_enabled"])
    # Keep reflector visible when the scenario enables it, even if one method
    # payload omits reflector optimization fields.
    if method_scene_config["reflector_enabled"]:
        if snapshot["reflector"]["target"] is None:
            fallback_target = _coerce_triplet(common_scene_config.get("focal_point"), fallback_z=1.5)
            if fallback_target is not None:
                snapshot["reflector"]["target"] = fallback_target
        if snapshot["reflector"]["u"] is None:
            snapshot["reflector"]["u"] = 0.5
        if snapshot["reflector"]["v"] is None:
            snapshot["reflector"]["v"] = 0.5

    method_row = method_summary_lookup.get(method_name, {})
    result_json_path = Path(str(method_row.get("result_json", artifacts_dir / f"{method_name}_results.json")))
    trace_json_path = artifacts_dir / f"{method_name}_iteration_trace.json"

    file_token = method_to_file_token.get(method_name, method_name)
    image_path = coverage_dir / f"{scenario_name}_{aps_tag}_{file_token}_coverage.png"

    try:
        rendered_path = _render_coverage_map(
            scene_config=method_scene_config,
            snapshot=snapshot,
            save_path=image_path,
            samples_per_tx=samples_per_tx,
            max_depth=max_depth,
            resolution=resolution,
            camera_position=camera_position,
            camera_look_at=camera_look_at,
        )
        if rendered_path is None:
            render_errors[method_name] = "Renderer returned no output path"
            continue

        sample_dbm, sample_metrics = _collect_rssi_samples(
            scene_config=method_scene_config,
            snapshot=snapshot,
            samples_per_tx=samples_per_tx,
            max_depth=max_depth,
        )
    except Exception as exc:
        render_errors[method_name] = f"{type(exc).__name__}: {exc}"
        continue

    if sample_dbm.size == 0:
        render_errors[method_name] = "No valid RSSI samples in computed radio map"
        continue

    for sample_index, sample_value in enumerate(sample_dbm.tolist(), start=1):
        all_sample_rows.append(
            {
                "scenario": scenario_name,
                "method": method_name,
                "seed": run_seed,
                "sample_index": sample_index,
                "rssi_dbm": float(sample_value),
            }
        )

    artifact_metrics = payload.get("best_physical_metrics", {})
    if not isinstance(artifact_metrics, Mapping):
        artifact_metrics = {}

    method_records[method_name] = {
        "method": method_name,
        "display_name": method_to_display_name.get(method_name, method_name),
        "run_seed": run_seed,
        "num_aps": int(snapshot["ap_count"]),
        "source_files": {
            "result_json": str(result_json_path),
            "trace_json": str(trace_json_path),
            "method_summary_row": method_row,
        },
        "snapshot": snapshot,
        "scene_config": method_scene_config,
        "artifact_metrics": dict(artifact_metrics),
        "sample_metrics": sample_metrics,
        "coverage_map_png": str(image_path),
    }

samples_df = pd.DataFrame(all_sample_rows)
if samples_df.empty:
    raise RuntimeError("No RSSI samples were collected from valid method snapshots")

samples_df = samples_df.sort_values(["method", "sample_index"], kind="stable")
samples_csv_path = output_root / f"qos_cdf_samples_{scenario_name}_{aps_tag}.csv"
samples_df.to_csv(samples_csv_path, index=False)

stats_df = (
    samples_df.groupby("method", as_index=False)
    .agg(
        scenario=("scenario", "first"),
        seed=("seed", "first"),
        valid_sample_count=("rssi_dbm", "size"),
        mean_rssi_dbm=("rssi_dbm", "mean"),
        median_rssi_dbm=("rssi_dbm", "median"),
        min_rssi_dbm=("rssi_dbm", "min"),
        p5_rssi_dbm=("rssi_dbm", lambda values: float(np.percentile(values, 5))),
        p95_rssi_dbm=("rssi_dbm", lambda values: float(np.percentile(values, 95))),
    )
)
stats_df["display_name"] = stats_df["method"].map(lambda name: method_to_display_name.get(name, name))
stats_df = stats_df[[
    "scenario",
    "method",
    "display_name",
    "seed",
    "valid_sample_count",
    "mean_rssi_dbm",
    "median_rssi_dbm",
    "min_rssi_dbm",
    "p5_rssi_dbm",
    "p95_rssi_dbm",
]]
stats_csv_path = output_root / f"qos_cdf_stats_{scenario_name}_{aps_tag}.csv"
stats_df.to_csv(stats_csv_path, index=False)

cdf_png_path = output_root / f"qos_cdf_{scenario_name}_{aps_tag}.png"
cdf_pdf_path = output_root / f"qos_cdf_{scenario_name}_{aps_tag}.pdf"
sla_threshold_dbm = -65.0

plt.figure(figsize=(10.5, 7.0))
for method_name in method_order:
    method_samples = samples_df.loc[samples_df["method"] == method_name, "rssi_dbm"].to_numpy(dtype=float)
    if method_samples.size == 0:
        continue
    x_values, y_values = _empirical_cdf(method_samples)
    plt.step(
        x_values,
        y_values,
        where="post",
        linewidth=2.2,
        color=method_colors.get(method_name, "#333333"),
        label=method_to_display_name.get(method_name, method_name),
    )

plt.axvline(sla_threshold_dbm, color="#444444", linestyle="--", linewidth=1.6, label="SLA = -65 dBm")
plt.title(f"QoS CDF ({scenario_name}, {aps_tag})")
plt.xlabel("RSSI (dBm)")
plt.ylabel("Empirical CDF")
plt.ylim(0.0, 1.0)
plt.xlim(float(samples_df["rssi_dbm"].min()) - 1.0, float(samples_df["rssi_dbm"].max()) + 1.0)
plt.grid(True, alpha=0.25)
plt.legend(frameon=True, ncol=2)
plt.tight_layout()
plt.savefig(cdf_png_path, dpi=220, bbox_inches="tight")
plt.savefig(cdf_pdf_path, bbox_inches="tight")
plt.show()
plt.close()

summary_payload = {
    "scenario": scenario_name,
    "aps_tag": aps_tag,
    "run_dir": str(run_dir),
    "artifacts_dir": str(artifacts_dir),
    "base_config_path": str(base_config_path),
    "scene_config": common_scene_config,
    "camera": {
        "position": list(camera_position),
        "look_at": list(camera_look_at),
    },
    "render_settings": {
        "samples_per_tx": samples_per_tx,
        "max_depth": max_depth,
        "resolution": list(resolution),
    },
    "method_order": method_order,
    "run_seed": run_seed,
    "method_summary_csv": str(method_summary_path),
    "missing_snapshot_methods": missing_snapshot_methods,
    "outputs": {
        "cdf_plot_png": str(cdf_png_path),
        "cdf_plot_pdf": str(cdf_pdf_path),
        "samples_csv": str(samples_csv_path),
        "stats_csv": str(stats_csv_path),
        "coverage_maps_dir": str(coverage_dir),
    },
    "methods": method_records,
    "render_errors": render_errors,
}

summary_json_path = output_root / f"qos_cdf_summary_{scenario_name}_{aps_tag}.json"
summary_json_path.write_text(json.dumps(summary_payload, indent=2, default=_json_default))

print(stats_df.to_string(index=False))
print()
print(f"CDF PNG: {cdf_png_path}")
print(f"CDF PDF: {cdf_pdf_path}")
print(f"Samples CSV: {samples_csv_path}")
print(f"Stats CSV: {stats_csv_path}")
print(f"Summary JSON: {summary_json_path}")
print(f"Coverage maps directory: {coverage_dir}")
if missing_snapshot_methods:
    print()
    print("Missing snapshots:", ", ".join(missing_snapshot_methods))
if render_errors:
    print()
    print("Render/sample errors:")
    for method_name, error_text in render_errors.items():
        print(f"- {method_name}: {error_text}")

scenario          method     display_name  seed  valid_sample_count  mean_rssi_dbm  median_rssi_dbm  min_rssi_dbm  p5_rssi_dbm  p95_rssi_dbm
  office          kmeans          k-means    42                2883     -64.299015       -64.769806   -123.547623   -85.649741    -41.460384
  office         memetic            GA+GD    42                2883     -65.397468       -65.762939   -126.326691   -87.627739    -42.271169
  office          pso_gd           PSO+GD    42                2881     -70.978895       -69.513489   -125.277985  -102.995270    -44.222031
  office          random           random    42                2877     -69.143156       -69.594284   -126.841309   -95.862395    -41.949017
  office       random_gd        random+GD    42                2883     -63.856510       -64.091484   -110.608124   -86.253201    -41.672576
  office weighted_kmeans weighted k-means    42                2883     -66.049519       -66.638405   -126.068054   -88.130484    -41.400829

CDF PNG: /ho

/tmp/ipykernel_8118/3866894860.py:328: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
